# POLSCI 3

## Week 10 Section Activity: Moneyball

This activity was inspired by *Moneyball* (the 2nd best movie).

In 2002, the Oakland A's had one of the smallest payrolls in baseball, but they still won 103 games. The story goes that they did it by buying hitters who got on base a lot, who were cheap because the rest of the league undervalued them. Can we see that in the data?

In this activity you'll practice **descriptive hypothesis testing** with `difference_in_means()` and **scatterplots** with a line of best fit.

The multiple choice answers can be checked by trying every letter, but it is for your own learning so please try to understand why the correct answer is what it is.

Please don't panic if you don't know anything about baseball, pretty much everything I know about baseball comes from watching Moneyball. You do not need to have any prior knowledge to complete this assignment.

### Quick Baseball 101
- **Plate Appearance (`pa`):** Every time a player steps up to bat.
- **On-Base Percentage (`obp`):** How often a player gets on base safely instead of getting out. This is all that matters (in Moneyball at least)
- **OPS (`ops`):** A single combined score for how good a hitter is (combining getting on base and hitting for power). Higher number = better hitter.

**Why Weighting Matters:**
Imagine a bench player who only plays in one game all year, gets 1 hit in 2 plate appearances, and ends the season with a `.500` OBP. Meanwhile, a starter plays 162 games with 600 plate appearances and has a `.380` OBP. If we calculate an *unweighted* average of player OBPs, that bench player counts just as much as the starter! Weighting by plate appearances (`pa`) ensures that players who actually played more have a proportional influence on the team's average.

In [ ]:
library(testthat)
library(estimatr)
library(ggplot2)

data <- read.csv('moneyball_batters.csv')
head(data)

Each row is one hitter's season with one team (a player who was traded mid-season appears once for each team). The data covers the 2001, 2002, and 2003 seasons. I dropped pitchers, since pitchers rarely bat in the American League (where the A's play) but do bat in the National League. Here is what the columns mean:

- `player`: the player's name
- `year`: the season (2001, 2002, or 2003)
- `team`: the team's abbreviation (the Oakland A's are `OAK`)
- `oakland`: `1` if the player was on the Oakland A's; `0` if the player was on any other team
- `pa`: the player's number of plate appearances that season (i.e., how many times they came up to bat)
- `ab`: the player's number of at-bats that season
- `obp`: on-base percentage, the share of plate appearances in which the player reached base (a walk counts, not just a hit)
- `slg`: slugging percentage, the average number of bases the player gets per at-bat (so home runs count much more than singles)
- `ops`: `obp` plus `slg`, a single number summarizing both
- `salary_millions`: the player's salary that season, in millions of dollars (missing for some bench players)
- `salary_group`: where the player's salary ranks among players in that same season: `"high"` is the top 25% of salaries, `"low"` is the bottom 50%, `"middle"` is in between, and `"unknown"` means no salary was listed

------

## Part 1: Did the 2002 A's get on base more?

**Question 1. Using only the 2002 season, estimate the difference in average `obp` between Oakland A's hitters (`oakland` = 1) and all other hitters (`oakland` = 0). Treat every player equally, no matter how many plate appearances they had.**

Save the output in `unweighted.2002`.

In [ ]:
# First, make a dataset with only the 2002 season
data.2002 <- subset(data, year == 2002)

# Then, estimate the (unweighted) difference in OBP between A's hitters and all other hitters
unweighted.2002 <- difference_in_means(obp ~ oakland, data = data.2002)

unweighted.2002 

In [ ]:
. = ottr::check("tests/q1.R")

------

**Question 2. Now estimate the same difference in average `obp` in 2002 (A's hitters, where `oakland` = 1, minus all other hitters, where `oakland` = 0), but this time give players more or less influence based on how much they played: weight each player by their number of plate appearances (`pa`).**

Save the output in `weighted.2002`.

In [ ]:
# Same as Question 1, but now weight each player by their plate appearances
weighted.2002 <- difference_in_means(obp ~ oakland, data = data.2002, weight = pa)

weighted.2002 

In [ ]:
. = ottr::check("tests/q2.R")

------

**Question 3. Compare your answers to Questions 1 and 2. Which of these best explains why the two estimates are so different?**

- `'a'`: The weighted estimate is wrong, because plate appearances are not survey weights.
- `'b'`: The unweighted estimate treats a bench player with a handful of plate appearances the same as a starter with 700, so a few bench players with unusual OBPs can swing a team's average. Weighting by `pa` lets each player count in proportion to how much they actually hit.
- `'c'`: Oakland's players had more plate appearances than other players, so weighting makes Oakland look worse.
- `'d'`: The two estimates should always be identical. If they aren't, there is a coding mistake.

Put your answer in the quotes below:

In [ ]:
q3.answer <- 'b'

------

**Question 4. Look at the _weighted_ estimate you made in Question 2, along with its $p$-value and 95% confidence interval. Which of these is the best conclusion?**

(One wrinkle: this data includes essentially every hitter in MLB that year, not a random sample of them. For this question, think of the $p$-value as asking whether a gap this big is surprising if Oakland's hitters and everyone else's were really the same and the gap was just luck of the draw.)

- `'a'`: Oakland hitters got on base significantly more often than other hitters in 2002.
- `'b'`: There is about a 58% chance that Oakland hitters got on base more often than other hitters in 2002.
- `'c'`: Oakland hitters got on base significantly less often than other hitters in 2002.
- `'d'`: Oakland hitters' OBP was only slightly higher in this data, and a gap this small would not be surprising even if Oakland and other hitters were truly the same, so we can't rule out that there is no difference.

Put your answer in the quotes below:

In [ ]:
q4.answer <- 'd'

------

## Part 2: A year before and a year after

Was 2002 special? Let's look at the seasons on either side of it.

**Question 5. For both 2001 and 2003, estimate the _weighted_ difference in average `obp` between Oakland A's hitters (`oakland` = 1) and all other hitters (`oakland` = 0), weighting each player by their plate appearances (`pa`), just like you did in Question 2.**

Save the 2001 output in `weighted.2001` and the 2003 output in `weighted.2003`.

In [ ]:
# 2001: subset to the season, then estimate the weighted difference
data.2001 <- subset(data, year == 2001)
weighted.2001 <- difference_in_means(obp ~ oakland, data = data.2001, weight = pa)

# 2003: same thing
data.2003 <- subset(data, year == 2003)
weighted.2003 <- difference_in_means(obp ~ oakland, data = data.2003, weight = pa)

weighted.2001 
weighted.2003

In [ ]:
. = ottr::check("tests/q5.R")

------

**Question 6. Looking at the weighted estimates for 2001 (Question 5), 2002 (Question 2), and 2003 (Question 5), which of these is true?**

- `'a'`: In 2002, but not in 2001 or 2003, Oakland hitters had a statistically significantly higher OBP than other hitters.
- `'b'`: In none of the three years can we rule out that Oakland hitters and other hitters had the same OBP.
- `'c'`: Oakland's OBP advantage over other hitters grew every year from 2001 to 2003.
- `'d'`: In 2003, Oakland hitters had a statistically significantly lower OBP than other hitters.

Put your answer in the quotes below:

In [ ]:
q6.answer <- 'b'

------

## Part 3: Did big salaries buy better hitting?

In *Moneyball*, the idea is that expensive stars are overvalued while cheaper players are undervalued. So let's see how much better the highest-paid hitters actually hit. For this part, use only the 2002 season.

**Question 7. Among 2002 hitters, estimate how much higher or lower `ops` is for `"high"` salary hitters than for `"low"` salary hitters (ignore the `"middle"` and `"unknown"` groups). Weight each player by their number of at-bats (`ab`).**

Your estimate should be the `"high"` group's weighted average OPS _minus_ the `"low"` group's weighted average OPS.

Save the output in `ops.by.salary`.

In [ ]:
# Make a dataset with only the 2002 season (this part is independent of the earlier parts)
data.2002 <- subset(data, year == 2002)

# Estimate the weighted difference in OPS: "high" minus "low"
# (condition1 is the comparison group, condition2 is the group we subtract it from)
ops.by.salary <- difference_in_means(ops ~ salary_group, data = data.2002,
                                     condition1 = "low", condition2 = "high",
                                     weight = ab)

ops.by.salary 

In [ ]:
. = ottr::check("tests/q7.R")

------

**Question 8. Which of these is the best interpretation of your results from Question 7?**

- `'a'`: In 2002, high-salary hitters had an average OPS about 0.11 higher than low-salary hitters, and a gap this large would be very surprising if the two groups were truly the same. This is a descriptive difference, not necessarily a causal effect of salary.
- `'b'`: Paying a hitter a high salary causes their OPS to rise by about 0.11.
- `'c'`: High-salary hitters had a slightly lower OPS than low-salary hitters, but not by a statistically significant amount.
- `'d'`: High-salary hitters got on base 11% more often than low-salary hitters.

Put your answer in the quotes below:

In [ ]:
q8.answer <- 'a'

------

## Part 4: Salary and on-base percentage, visualized

Differences between groups are crude. Let's look at the whole relationship between salary and OBP instead. For this part, use only the 2002 season.

**Question 9. Make a scatterplot of 2002 hitters where salary (`salary_millions`) is on the x-axis and `obp` is on the y-axis.**

Do _not_ include a line of best fit yet. Save the graph in `salary.graph`.

In [ ]:
# Make a dataset with only the 2002 season
data.2002 <- subset(data, year == 2002)

# Scatterplot with salary on the x axis and OBP on the y axis
salary.graph <- qplot(salary_millions, obp, data = data.2002)

salary.graph # Let's look at the graph.

In [ ]:
. = ottr::check("tests/q9.R")

------

**Question 10. Add a line of best fit to the graph you made in Question 9.**

Save the new graph in `salary.graph.with.line`.

In [ ]:
# Same as Question 9, plus a line of best fit
salary.graph.with.line <- qplot(salary_millions, obp, data = data.2002) + geom_smooth(method = 'lm')

salary.graph.with.line # Let's look at the graph.

In [ ]:
. = ottr::check("tests/q10.R")

------

**Question 11. Looking at the graph you made in Question 10, what OBP does the line of best fit predict for a hitter who is paid $5 million?**

(Make your best guess from looking at the graph. This question allows room for error on either side of the right answer.)

**Enter your answer as a decimal rounded to three places. For example, if you wanted to answer .300, you would type `my.best.guess <- 0.300`.**

In [ ]:
my.best.guess <- 0.336

In [ ]:
. = ottr::check("tests/q11.R")

------

**Question 12. Which of these best describes what your graph shows about salary and OBP in 2002?**

- `'a'`: Hitters who were paid more tended to have a lower OBP.
- `'b'`: There is no relationship between salary and OBP.
- `'c'`: Hitters who were paid more tended to have a somewhat higher OBP on average, but there are lots of cheap hitters with a high OBP and lots of expensive hitters with a low OBP.
- `'d'`: Salary perfectly predicts OBP.

Put your answer in the quotes below:

In [ ]:
q12.answer <- 'c'

----